# Objetivo_2

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import altair as alt
import datos
import plotly.graph_objects as go
import warnings
import plotly.subplots as sp

warnings.simplefilter(action='ignore', category=FutureWarning)

tipo de afiliación, lugar de nacimiento/procedencia, escolaridad,   antecedente personal de dislipidemia o hipercolesterolemia, de hipertensión, de infarto de miocardio de diabetes, insuficiencia cardiaca congestiva, nefropatía, arritmia, válvula, angeopatía, fumador y bebedor.  

+ IMC_calculada
+ perimetro_abdominal
+ presion_arterial_sistolica
+ presion_arterial_diastolica

+ <19.9 bajo peso, 20-24.9 normo peso, 25-29.9 sobrepeso, >=30 obesidad

+ normal (diastólica <80 y sistólica <120), 
 elevada (sistólica 120-129 y diastólica <80),
 alta nivel 1 (sistólica 130-139 o diastólica 80-89), 
 alta nivel 2 (sistólica >=140, diastólica >=90), 
 crisis hipertensiva (>180 diastólica y/o >120 sistólica)
+ aumentado >94 hombres, >80 mujeres; incremento sustancial >102 hombres y >88 en mujeres

### IMC

In [5]:

intervalos = {
    'IMC': [0, 19.9, 24.9, 29.9, 40, max(datos.datos['IMC'])+1]
}

# Agrupaciones disponibles
agrupaciones = [
    'sexo',
    'tipo_de_afiliacion',
    'escolaridad',
    'diabetes mellitus',
    'hipercolesterolemia',
    'dislipidemia',
    'no_diabeticos_POC_hba1c(%)_controlado',
    'diabeticos_POC_hba1c(%)_controlado', 
    'bajo_peso',
    'normo_peso',
    'sobrepeso',
    'obesidad',
    'perimetro_abdominal_normal',
    'perimetro_abdominal_aumentado',
    'perimetro_abdominal_incremento_sustancial',
    'tension_normal',
    'tension_elevada',
    'tension_alta_nivel_1',
    'tension_alta_nivel_2'
]
def plot_all():
    fig = px.box(datos.datos, y='IMC', title='Boxplot crudo del IMC')
    fig.update_layout(xaxis_title='IMC')
    fig.show()
    fig = px.histogram(datos.datos, x='IMC', nbins=10, title='Histograma crudo del IMC', marginal='rug')
    fig.update_layout(xaxis_title='IMC')
    fig.show()

    # Mostrar tabla descriptiva general
    tabla = datos.datos['IMC'].describe().reset_index()
    tabla.columns = ['Estadística', 'Valor']
    print(tabla)
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla[col] for col in tabla.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas del IMC')
    fig.show()

    # Graficar IMC por cada agrupación, excluyendo 'lugar_de_procedencia' y 'lugar_de_nacimiento'
    for agrupacion in agrupaciones:
        if agrupacion in ['tipo_de_afiliacion','escolaridad']:
            continue

        # Estadísticas descriptivas por agrupación
        tabla2 = datos.datos.groupby(agrupacion)['IMC'].describe().reset_index()
        tabla2 = tabla2.rename(columns={'mean': 'Media', 'std': 'Desviación estándar', '50%': 'Mediana', 'min': 'Mínimo', 'max': 'Máximo'})
        print(tabla2)
        fig = go.Figure(data=[go.Table(
            header=dict(values=list(tabla2.columns), fill_color='paleturquoise', align='left'),
            cells=dict(values=[tabla2[col] for col in tabla2.columns], fill_color='lavender', align='left')
        )])
        fig.update_layout(title_text=f'Estadísticas Descriptivas del IMC por {agrupacion}')
        fig.show()

        # Boxplot y histograma por agrupación
        fig = px.box(datos.datos, x='IMC', y=agrupacion, title=f'Boxplot del IMC por {agrupacion}')
        fig.update_layout(xaxis_title='IMC')
        fig.show()

        fig = px.histogram(datos.datos, x='IMC', color=agrupacion, nbins=10, title=f'Histograma del IMC por {agrupacion}', marginal='rug')
        fig.update_layout(xaxis_title='IMC')
        fig.show()


    
    datos.datos['IMC_intervalo'] = pd.cut(datos.datos['IMC'], bins=[0, 19.9, 24.9, 29.9, 40.0, 50.66])
    datos.datos['IMC_intervalo'] = datos.datos['IMC_intervalo'].astype(str)

        
    tabla3 = datos.datos.groupby('IMC_intervalo')['IMC'].describe().reset_index()
    tabla3 = tabla3.rename(columns={'mean': 'Media', 'std': 'Desviación estándar', '50%': 'Mediana', 'min': 'Mínimo', 'max': 'Máximo'})
    display(tabla3)
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla3.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla3[col] for col in tabla3.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas del IMC por Intervalos')
    fig.show()

    orden_intervalos = ['(0.0, 19.9]', '(19.9, 24.9]', '(24.9, 29.9]', '(29.9, 40.0]', '(40.0, 50.66]']
    fig = px.box(datos.datos, x=pd.Categorical(datos.datos['IMC_intervalo'], categories=orden_intervalos, ordered=True), 
                y='IMC', title="Boxplot del IMC por Intervalos", color_discrete_sequence=["blue"])
    fig.update_layout(xaxis_title='Intervalo IMC', yaxis_title='IMC', xaxis={'categoryorder':'array', 'categoryarray':orden_intervalos})
    fig.show()
plot_all()
for agrupacion in ['lugar_de_procedencia', 'lugar_de_nacimiento']:
    tabla_freq = datos.datos[agrupacion].value_counts().reset_index()
    tabla_freq.columns = [agrupacion, 'Frecuencia']
    print(tabla_freq)
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla_freq.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla_freq[col] for col in tabla_freq.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text=f'Tabla de Frecuencias para {agrupacion}')
    fig.show()


  Estadística       Valor
0       count  952.000000
1        mean   26.548866
2         std    4.354945
3         min   15.380000
4         25%   23.555000
5         50%   26.070000
6         75%   29.075000
7         max   49.660000


        sexo  count      Media  Desviación estándar  Mínimo      25%  Mediana  \
0   femenino  330.0  27.039758             5.012477   15.38  23.3775   26.480   
1  masculino  622.0  26.288424             3.941689   16.64  23.6700   25.915   

      75%  Máximo  
0  30.205   45.94  
1  28.730   49.66  


  diabetes mellitus  count      Media  Desviación estándar  Mínimo      25%  \
0                no  646.0  26.471362             4.382571   16.02  23.4175   
1                si  306.0  26.712484             4.298567   15.38  23.7025   

   Mediana     75%  Máximo  
0   26.060  28.945   49.66  
1   26.115  29.590   47.30  


  hipercolesterolemia  count      Media  Desviación estándar  Mínimo    25%  \
0                  no  939.0  26.573621             4.354615   15.38  23.62   
1                  si   13.0  24.760769             4.154418   18.51  22.41   

   Mediana    75%  Máximo  
0    26.08  29.09   49.66  
1    24.93  28.73   30.25  


  dislipidemia  count      Media  Desviación estándar  Mínimo    25%  Mediana  \
0           no  664.0  26.434292             4.373007   15.38  23.49   26.050   
1           si  288.0  26.813021             4.308952   16.64  23.66   26.195   

     75%  Máximo  
0  28.90   49.66  
1  29.65   40.30  


...

  tension_alta_nivel_1  count      Media  Desviación estándar  Mínimo     25%  \
0                   no  511.0  26.228669             4.351604   15.38  23.155   
1                   si  441.0  26.919887             4.334204   17.01  23.930   

   Mediana    75%  Máximo  
0    25.85  28.76   49.66  
1    26.45  29.55   47.30  


  tension_alta_nivel_2  count      Media  Desviación estándar  Mínimo     25%  \
0                   no  819.0  26.416068             4.217063   15.38  23.505   
1                   si  133.0  27.366617             5.064621   16.64  23.950   

   Mediana    75%  Máximo  
0    25.94  29.00   47.30  
1    26.71  30.05   49.66  


,IMC_intervalo,count,Media,Desviación estándar,Mínimo,25%,Mediana,75%,Máximo
0,"(0.0, 19.9]",41.0,18.706829,1.092901,15.38,18.5100,19.02,19.47,19.85
1,"(19.9, 24.9]",319.0,22.945486,1.314399,19.99,22.0450,23.07,24.02,24.90
2,"(24.9, 29.9]",401.0,27.162294,1.426840,24.91,25.9000,27.02,28.36,29.90
3,"(29.9, 40.0]",185.0,32.582919,2.373368,29.93,30.6300,31.99,33.78,39.90
4,"(40.0, 50.66]",6.0,44.668333,3.695004,40.30,41.5775,45.00,46.96,49.66
5,nan,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


   lugar_de_procedencia  Frecuencia
0                ibague         243
1               armenia         187
2                 yopal         149
3                bogota         118
4               acacias         105
5               calarca          32
6              quimbaya          21
7            montenegro          12
8               tebaida           9
9              circasia           8
10            barcelona           4
11              aguazul           4
12             filandia           3
13               genova           3
14        corregimiento           3
15              espinal           3
16              salento           3
17              cordoba           2
18         barranquilla           2
19        villavicencio           2
20               vereda           2
21           caicedonia           2
22               alcala           2
23             icononzo           1
24          procedencia           1
25                pijao           1
26               ovando     

    lugar_de_nacimiento  Frecuencia
0                ibague          92
1               armenia          73
2                bogota          70
3               calarca          24
4                 yopal          23
..                  ...         ...
299             gigante           1
300               lloro           1
301          roldanillo           1
302               pacho           1
303            guateque           1

[304 rows x 2 columns]


### Perimetro abdominal

In [6]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Definir los intervalos para perimetro_abdominal
intervalos = {
    'masculino': [0, 94, 102, max(datos.datos['perimetro_abdominal'])],
    'femenino': [0, 80, 88, max(datos.datos['perimetro_abdominal'])] 
}
agrupaciones = [
        'sexo', 'tipo_de_afiliacion', 'lugar_de_procedencia',
        'lugar_de_nacimiento', 'escolaridad', 'diabetes mellitus',
        'hipercolesterolemia', 'dislipidemia'
    ]

def plot_all():
    # Graficar perimetro_abdominal crudo
    fig = px.box(datos.datos, y='perimetro_abdominal', title='Boxplot crudo de perimetro_abdominal')
    fig.update_layout(yaxis_title='Perímetro Abdominal')
    fig.show()

    fig = px.histogram(datos.datos, x='perimetro_abdominal', nbins=10, title='Histograma crudo de perimetro_abdominal', marginal='rug')
    fig.update_layout(xaxis_title='Perímetro Abdominal')
    fig.show()

    # Mostrar tabla descriptiva general verticalmente
    tabla = datos.datos['perimetro_abdominal'].describe().reset_index()
    tabla.columns = ['Estadística', 'Valor']
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla[col] for col in tabla.columns], fill_color='lavender', align='left')
    )])
    
    fig.update_layout(title_text='Estadísticas Descriptivas del perimetro_abdominal')
    fig.show()
    print(tabla)
    # Agrupaciones
    agrupaciones = [
        'sexo', 'tipo_de_afiliacion',  'escolaridad', 'diabetes mellitus',
        'hipercolesterolemia', 'dislipidemia', 'no_diabeticos_POC_hba1c(%)_controlado',
    'diabeticos_POC_hba1c(%)_controlado', 
    'bajo_peso',
    'normo_peso',
    'sobrepeso',
    'obesidad',
    'perimetro_abdominal_normal',
    'perimetro_abdominal_aumentado',
    'perimetro_abdominal_incremento_sustancial',
    'tension_normal',
    'tension_elevada',
    'tension_alta_nivel_1',
    'tension_alta_nivel_2'
    ]
    # 'lugar_de_procedencia',
    #    'lugar_de_nacimiento',

    for agrupacion in agrupaciones:
        if agrupacion not in ['lugar_de_procedencia', 'lugar_de_nacimiento','tipo_de_afiliacion','escolaridad']:
            # Graficar perimetro_abdominal por la variable de agrupación
            fig = px.box(datos.datos, x='perimetro_abdominal', y=agrupacion, title=f'Boxplot de perimetro_abdominal por {agrupacion}')
            fig.update_layout(xaxis_title='Perímetro Abdominal')
            fig.show()

            fig = px.histogram(datos.datos, x='perimetro_abdominal', color=agrupacion, nbins=10, title=f'Histograma de perimetro_abdominal por {agrupacion}', marginal='rug')
            fig.update_layout(xaxis_title='Perímetro Abdominal')
            fig.show()

            # Mostrar tabla descriptiva por grupo verticalmente
            tabla2 = datos.datos.groupby(agrupacion)['perimetro_abdominal'].describe().reset_index()
            tabla2 = tabla2.rename(columns={'mean': 'Media', 'std': 'Desviación estándar', '50%': 'Mediana', 'min': 'Mínimo', 'max': 'Máximo'})
            print(tabla2)
            fig = go.Figure(data=[go.Table(
                header=dict(values=list(tabla2.columns), fill_color='paleturquoise', align='left'),
                cells=dict(values=[tabla2[col] for col in tabla2.columns], fill_color='lavender', align='left')
            )])
            fig.update_layout(title_text=f'Estadísticas Descriptivas del perimetro_abdominal por {agrupacion}')
            fig.show()
    hom = datos.datos[datos.datos['sexo'] == 'masculino']
    muj = datos.datos[datos.datos['sexo'] == 'femenino']
    hom['Intervalo'] = pd.cut(hom['perimetro_abdominal'], bins=[0, 94, 102, hom['perimetro_abdominal'].max()], labels=['Normal', 'Aumentado', 'Incremento Sustancial'])
    muj['Intervalo'] = pd.cut(muj['perimetro_abdominal'], bins=[0, 80, 88, muj['perimetro_abdominal'].max()], labels=['Normal', 'Aumentado', 'Incremento Sustancial'])
    fig_hombres = go.Figure()
    fig_hombres.add_trace(go.Box(y=hom[hom['Intervalo'] == 'Normal']['perimetro_abdominal'], name='Normal', marker_color='green'))
    fig_hombres.add_trace(go.Box(y=hom[hom['Intervalo'] == 'Aumentado']['perimetro_abdominal'], name='Aumentado', marker_color='orange'))
    fig_hombres.add_trace(go.Box(y=hom[hom['Intervalo'] == 'Incremento Sustancial']['perimetro_abdominal'], name='Incremento Sustancial', marker_color='red'))
    fig_hombres.update_layout(title="Gráficos de Caja del Perímetro Abdominal - Hombres", yaxis_title="Perímetro Abdominal (cm)", boxmode='group', xaxis_title="Intervalos")
    fig_hombres.show()
    fig_mujeres = go.Figure()
    fig_mujeres.add_trace(go.Box(y=muj[muj['Intervalo'] == 'Normal']['perimetro_abdominal'], name='Normal', marker_color='blue'))
    fig_mujeres.add_trace(go.Box(y=muj[muj['Intervalo'] == 'Aumentado']['perimetro_abdominal'], name='Aumentado', marker_color='purple'))
    fig_mujeres.add_trace(go.Box(y=muj[muj['Intervalo'] == 'Incremento Sustancial']['perimetro_abdominal'], name='Incremento Sustancial', marker_color='darkred'))
    fig_mujeres.update_layout(title="Gráficos de Caja del Perímetro Abdominal - Mujeres", yaxis_title="Perímetro Abdominal (cm)", boxmode='group', xaxis_title="Intervalos")
    fig_mujeres.show()
plot_all()

  Estadística       Valor
0       count  952.000000
1        mean   93.809779
2         std   11.684993
3         min    1.010000
4         25%   87.000000
5         50%   94.000000
6         75%  101.000000
7         max  151.000000


        sexo  count      Media  Desviación estándar  Mínimo   25%  Mediana  \
0   femenino  330.0  91.235788            13.310194    1.01  83.0     90.5   
1  masculino  622.0  95.175402            10.480886   30.00  88.0     95.0   

     75%  Máximo  
0  100.0   151.0  
1  102.0   140.0  


  diabetes mellitus  count      Media  Desviación estándar  Mínimo   25%  \
0                no  646.0  93.183916            11.733874    1.01  86.0   
1                si  306.0  95.131046            11.488463   56.00  88.0   

   Mediana     75%  Máximo  
0     93.0  100.75   151.0  
1     95.0  102.00   140.0  


  hipercolesterolemia  count      Media  Desviación estándar  Mínimo   25%  \
0                  no  939.0  93.856134            11.673310    1.01  87.0   
1                  si   13.0  90.461538            12.527406   70.00  86.0   

   Mediana    75%  Máximo  
0     94.0  101.0   151.0  
1     91.0   98.0   117.0  


  dislipidemia  count      Media  Desviación estándar  Mínimo   25%  Mediana  \
0           no  664.0  93.625316            11.763525    1.01  87.0     93.0   
1           si  288.0  94.235069            11.510891   36.00  88.0     94.0   

     75%  Máximo  
0  101.0   151.0  
1  102.0   127.0  


...

  tension_normal  count      Media  Desviación estándar  Mínimo   25%  \
0             no  721.0  94.725811            11.814372    1.01  88.0   
1             si  231.0  90.950649            10.806226   64.00  84.0   

   Mediana    75%  Máximo  
0     95.0  102.0   151.0  
1     91.0   98.0   120.0  


  tension_elevada  count      Media  Desviación estándar  Mínimo   25%  \
0              no  805.0  93.770571            11.924410    1.01  87.0   
1              si  147.0  94.024490            10.309794   68.00  87.5   

   Mediana    75%  Máximo  
0     94.0  101.0   151.0  
1     94.0  101.5   120.0  


  tension_alta_nivel_1  count      Media  Desviación estándar  Mínimo   25%  \
0                   no  511.0  92.935616            11.030758   64.00  86.0   
1                   si  441.0  94.822698            12.335287    1.01  88.0   

   Mediana    75%  Máximo  
0     92.0  100.0   130.0  
1     95.0  102.0   151.0  


  tension_alta_nivel_2  count      Media  Desviación estándar  Mínimo   25%  \
0                   no  819.0  93.587314            11.682113    1.01  87.0   
1                   si  133.0  95.179699            11.653006   65.00  88.0   

   Mediana    75%  Máximo  
0     94.0  101.0   151.0  
1     96.0  103.0   130.0  


C:\Users\loren\AppData\Local\Temp\ipykernel_16064\1180647437.py:80: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\loren\AppData\Local\Temp\ipykernel_16064\1180647437.py:81: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



### POC_hba1

In [7]:

intervalos = {
    'masculino': [0, 120, 130, 139, max(datos.datos['presion_arterial_sistolica'])],
    'femenino': [0, 120, 130, 139, max(datos.datos['presion_arterial_sistolica'])]
}

def plot_all():
    fig = px.box(datos.datos, y='presion_arterial_sistolica', title='Boxplot crudo de presión arterial sistólica')
    fig.update_layout(yaxis_title='Presión Arterial Sistólica')
    fig.show()

    fig = px.histogram(datos.datos, x='presion_arterial_sistolica', nbins=10, title='Histograma crudo de presión arterial sistólica', marginal='rug')
    fig.update_layout(xaxis_title='Presión Arterial Sistólica')
    fig.show()

    tabla = datos.datos['presion_arterial_sistolica'].describe().reset_index()
    tabla.columns = ['Estadística', 'Valor']
    print(tabla)
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla[col] for col in tabla.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas de presión arterial sistólica')
    fig.show()

    agrupaciones = ['sexo', 'tipo_de_afiliacion', 'lugar_de_procedencia', 'lugar_de_nacimiento', 'escolaridad', 'diabetes mellitus', 'hipercolesterolemia', 'dislipidemia']
    for agrupacion in agrupaciones:
        if agrupacion in ['lugar_de_procedencia', 'lugar_de_nacimiento', 'tipo_de_afiliacion', 'escolaridad']:
            continue
        else:
            fig = px.box(datos.datos, x='presion_arterial_sistolica', y=agrupacion, title=f'Boxplot de presión arterial sistólica por {agrupacion}')
            fig.update_layout(xaxis_title='Presión Arterial Sistólica', yaxis_title=agrupacion)
            fig.show()

            fig = px.histogram(datos.datos, x='presion_arterial_sistolica', color=agrupacion, nbins=10, title=f'Histograma de presión arterial sistólica por {agrupacion}', marginal='rug')
            fig.update_layout(xaxis_title='Presión Arterial Sistólica')
            fig.show()

            tabla_agrupacion = datos.datos.groupby(agrupacion)['presion_arterial_sistolica'].describe().reset_index()
            tabla_agrupacion = tabla_agrupacion.rename(columns={'mean': 'Media', 'std': 'Desviación estándar', '50%': 'Mediana', 'min': 'Mínimo', 'max': 'Máximo'})
            print(tabla_agrupacion)
            fig = go.Figure(data=[go.Table(
                header=dict(values=list(tabla_agrupacion.columns), fill_color='paleturquoise', align='left'),
                cells=dict(values=[tabla_agrupacion[col] for col in tabla_agrupacion.columns], fill_color='lavender', align='left')
            )])
            fig.update_layout(title_text=f'Estadísticas Descriptivas de presión arterial sistólica por {agrupacion}')
            fig.show()

    datos.datos['presion_arterial_sistolica_intervalo'] = pd.cut(datos.datos['presion_arterial_sistolica'], intervalos['masculino'], right=False).astype(str)
    tabla_intervalos = datos.datos.groupby('presion_arterial_sistolica_intervalo')['presion_arterial_sistolica'].describe().reset_index()
    tabla_intervalos.columns = ['Intervalo', 'Cuenta', 'Media', 'Desviación estándar', 'Mínimo', '25%', 'Mediana', '75%', 'Máximo']
    print(tabla_intervalos)
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla_intervalos.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla_intervalos[col] for col in tabla_intervalos.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas por Intervalos de presión arterial sistólica')
    fig.show()

    intervalos_masculino = [0, 120, 130, 139, float('inf')]
    labels = ['[0, 120)', '[120, 130)', '[130, 139)', '[139, inf)']
    datos.datos['presion_arterial_sistolica_intervalo'] = pd.cut(datos.datos['presion_arterial_sistolica'], bins=intervalos_masculino, labels=labels, right=False, include_lowest=True)
    datos_filtrados = datos.datos.dropna(subset=['presion_arterial_sistolica_intervalo'])
    datos_filtrados['presion_arterial_sistolica_intervalo'] = pd.Categorical(datos_filtrados['presion_arterial_sistolica_intervalo'], categories=labels, ordered=True)
    fig = px.box(datos_filtrados, x='presion_arterial_sistolica_intervalo', y='presion_arterial_sistolica', title='Boxplot de Presión Arterial Sistólica por Intervalos', color_discrete_sequence=["blue"])
    fig.update_layout(xaxis_title='Intervalo', yaxis_title='Presión Arterial Sistólica', xaxis={'categoryorder': 'array', 'categoryarray': labels})
    fig.show()

plot_all()

  Estadística       Valor
0       count  952.000000
1        mean  126.969538
2         std   16.547104
3         min   90.000000
4         25%  117.000000
5         50%  125.000000
6         75%  135.000000
7         max  198.000000


        sexo  count       Media  Desviación estándar  Mínimo    25%  Mediana  \
0   femenino  330.0  128.557576            16.356348    97.0  120.0    129.5   
1  masculino  622.0  126.127010            16.598846    90.0  115.0    124.0   

      75%  Máximo  
0  137.75   190.0  
1  135.00   198.0  


  diabetes mellitus  count       Media  Desviación estándar  Mínimo    25%  \
0                no  646.0  126.969040            16.714445    90.0  117.0   
1                si  306.0  126.970588            16.215226    93.0  117.0   

   Mediana     75%  Máximo  
0    124.5  135.75   198.0  
1    128.0  135.00   180.0  


  hipercolesterolemia  count       Media  Desviación estándar  Mínimo    25%  \
0                  no  939.0  127.076677            16.575294    90.0  117.0   
1                  si   13.0  119.230769            12.564194   103.0  110.0   

   Mediana    75%  Máximo  
0    125.0  135.5   198.0  
1    117.0  120.0   146.0  


  dislipidemia  count       Media  Desviación estándar  Mínimo    25%  \
0           no  664.0  126.496988            17.035142    90.0  115.0   
1           si  288.0  128.059028            15.335959    90.0  120.0   

   Mediana     75%  Máximo  
0    124.5  135.00   190.0  
1    128.0  138.25   198.0  


        Intervalo  Cuenta       Media  Desviación estándar  Mínimo    25%  \
0    [0.0, 120.0)   265.0  108.916981             6.400217    90.0  105.0   
1  [120.0, 130.0)   267.0  122.318352             3.002468   120.0  120.0   
2  [130.0, 139.0)   211.0  131.924171             2.698136   130.0  130.0   
3  [139.0, 198.0)   208.0  150.572115            11.507804   139.0  140.0   
4             nan     1.0  198.000000                  NaN   198.0  198.0   

   Mediana    75%  Máximo  
0    110.0  113.0   119.0  
1    120.0  125.0   129.0  
2    130.0  134.0   138.0  
3    147.5  159.0   190.0  
4    198.0  198.0   198.0  


C:\Users\loren\AppData\Local\Temp\ipykernel_16064\1322565378.py:63: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



### Presion Arterial

####   Presion arterial sistolica

In [8]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Definición de intervalos para presión arterial sistólica
intervalos = {
    'masculino': [0, 120, 130, 139, max(datos.datos['presion_arterial_sistolica'])],
    'femenino': [0, 120, 130, 139, max(datos.datos['presion_arterial_sistolica'])]
}

def plot_all():
    # Boxplot crudo
    fig = px.box(datos.datos, y='presion_arterial_sistolica', title='Boxplot crudo de presión arterial sistólica')
    fig.update_layout(yaxis_title='Presión Arterial Sistólica')
    fig.show()

    # Histograma crudo
    fig = px.histogram(datos.datos, x='presion_arterial_sistolica', nbins=10, 
                       title='Histograma crudo de presión arterial sistólica', marginal='rug')
    fig.update_layout(xaxis_title='Presión Arterial Sistólica')
    fig.show()

    # Tabla descriptiva general
    tabla = datos.datos['presion_arterial_sistolica'].describe().reset_index()
    tabla.columns = ['Estadística', 'Valor']
    print(tabla)
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla[col] for col in tabla.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas de presión arterial sistólica')
    fig.show()

    # Tablas descriptivas y gráficos por agrupaciones
    agrupaciones = [
        'sexo', 'tipo_de_afiliacion',
        'escolaridad', 'diabetes mellitus', 'hipercolesterolemia', 'dislipidemia',
        'no_diabeticos_POC_hba1c(%)_controlado',
    'diabeticos_POC_hba1c(%)_controlado', 
    'bajo_peso',
    'normo_peso',
    'sobrepeso',
    'obesidad',
    'perimetro_abdominal_normal',
    'perimetro_abdominal_aumentado',
    'perimetro_abdominal_incremento_sustancial',
    'tension_normal',
    'tension_elevada',
    'tension_alta_nivel_1',
    'tension_alta_nivel_2'
    ]
    for agrupacion in agrupaciones:
        if agrupacion in ['lugar_de_procedencia', 'lugar_de_nacimiento', 'tipo_de_afiliacion', 'escolaridad']:
            continue  # Omitir estas agrupaciones específicas
        else:
            # Boxplot por agrupación
            fig = px.box(datos.datos, x='presion_arterial_sistolica', y=agrupacion, 
                         title=f'Boxplot de presión arterial sistólica por {agrupacion}')
            fig.update_layout(xaxis_title='Presión Arterial Sistólica', yaxis_title=agrupacion)
            fig.show()

            # Histograma por agrupación
            fig = px.histogram(datos.datos, x='presion_arterial_sistolica', color=agrupacion, nbins=10, 
                               title=f'Histograma de presión arterial sistólica por {agrupacion}', marginal='rug')
            fig.update_layout(xaxis_title='Presión Arterial Sistólica')
            fig.show()

            # Tabla descriptiva por agrupación
            tabla_agrupacion = datos.datos.groupby(agrupacion)['presion_arterial_sistolica'].describe().reset_index()
            
            tabla_agrupacion = tabla_agrupacion.rename(columns={
                'mean': 'Media', 'std': 'Desviación estándar', '50%': 'Mediana', 
                'min': 'Mínimo', 'max': 'Máximo'
            })
            print(tabla_agrupacion)
            fig = go.Figure(data=[go.Table(
                header=dict(values=list(tabla_agrupacion.columns), fill_color='paleturquoise', align='left'),
                cells=dict(values=[tabla_agrupacion[col] for col in tabla_agrupacion.columns], fill_color='lavender', align='left')
            )])
            fig.update_layout(title_text=f'Estadísticas Descriptivas de presión arterial sistólica por {agrupacion}')
            fig.show()

    # Tabla descriptiva por intervalos
    datos.datos['presion_arterial_sistolica_intervalo'] = pd.cut(
        datos.datos['presion_arterial_sistolica'],
        intervalos['masculino'],  # Usar intervalos para masculino como ejemplo
        right=False
    ).astype(str)
    tabla_intervalos = datos.datos.groupby('presion_arterial_sistolica_intervalo')['presion_arterial_sistolica'].describe().reset_index()
    tabla_intervalos.columns = ['Intervalo', 'Cuenta', 'Media', 'Desviación estándar', 'Mínimo', '25%', 'Mediana', '75%', 'Máximo']
    print(tabla_intervalos)
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla_intervalos.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla_intervalos[col] for col in tabla_intervalos.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas por Intervalos de presión arterial sistólica')
    fig.show()

    # Boxplot por intervalos categóricos
    intervalos_masculino = [0, 120, 130, 139, float('inf')]
    labels = ['[0, 120)', '[120, 130)', '[130, 139)', '[139, inf)']
    datos.datos['presion_arterial_sistolica_intervalo'] = pd.cut(
        datos.datos['presion_arterial_sistolica'],
        bins=intervalos_masculino,
        labels=labels,
        right=False,
        include_lowest=True
    )
    datos_filtrados = datos.datos.dropna(subset=['presion_arterial_sistolica_intervalo'])
    datos_filtrados['presion_arterial_sistolica_intervalo'] = pd.Categorical(
        datos_filtrados['presion_arterial_sistolica_intervalo'],
        categories=labels,
        ordered=True
    )
    fig = px.box(
        datos_filtrados, 
        x='presion_arterial_sistolica_intervalo', 
        y='presion_arterial_sistolica', 
        title='Boxplot de Presión Arterial Sistólica por Intervalos', 
        color_discrete_sequence=["blue"]
    )
    fig.update_layout(
        xaxis_title='Intervalo', 
        yaxis_title='Presión Arterial Sistólica',
        xaxis={'categoryorder': 'array', 'categoryarray': labels}
    )
    fig.show()

plot_all()



  Estadística       Valor
0       count  952.000000
1        mean  126.969538
2         std   16.547104
3         min   90.000000
4         25%  117.000000
5         50%  125.000000
6         75%  135.000000
7         max  198.000000


        sexo  count       Media  Desviación estándar  Mínimo    25%  Mediana  \
0   femenino  330.0  128.557576            16.356348    97.0  120.0    129.5   
1  masculino  622.0  126.127010            16.598846    90.0  115.0    124.0   

      75%  Máximo  
0  137.75   190.0  
1  135.00   198.0  


  diabetes mellitus  count       Media  Desviación estándar  Mínimo    25%  \
0                no  646.0  126.969040            16.714445    90.0  117.0   
1                si  306.0  126.970588            16.215226    93.0  117.0   

   Mediana     75%  Máximo  
0    124.5  135.75   198.0  
1    128.0  135.00   180.0  


  hipercolesterolemia  count       Media  Desviación estándar  Mínimo    25%  \
0                  no  939.0  127.076677            16.575294    90.0  117.0   
1                  si   13.0  119.230769            12.564194   103.0  110.0   

   Mediana    75%  Máximo  
0    125.0  135.5   198.0  
1    117.0  120.0   146.0  


  dislipidemia  count       Media  Desviación estándar  Mínimo    25%  \
0           no  664.0  126.496988            17.035142    90.0  115.0   
1           si  288.0  128.059028            15.335959    90.0  120.0   

   Mediana     75%  Máximo  
0    124.5  135.00   190.0  
1    128.0  138.25   198.0  


...

  tension_elevada  count       Media  Desviación estándar  Mínimo    25%  \
0              no  805.0  127.795031            17.825271    90.0  114.0   
1              si  147.0  122.448980             3.068139   120.0  120.0   

   Mediana    75%  Máximo  
0    130.0  139.0   198.0  
1    121.0  125.0   129.0  


  tension_alta_nivel_1  count       Media  Desviación estándar  Mínimo    25%  \
0                   no  511.0  123.481409             19.09715    90.0  110.0   
1                   si  441.0  131.011338             11.77159   100.0  124.0   

   Mediana    75%  Máximo  
0    120.0  129.0   198.0  
1    130.0  136.0   190.0  


  tension_alta_nivel_2  count       Media  Desviación estándar  Mínimo    25%  \
0                   no  819.0  123.152625            13.459707    90.0  114.5   
1                   si  133.0  150.473684            14.246519   110.0  142.0   

   Mediana    75%  Máximo  
0    121.0  130.0   190.0  
1    150.0  160.0   198.0  


        Intervalo  Cuenta       Media  Desviación estándar  Mínimo    25%  \
0    [0.0, 120.0)   265.0  108.916981             6.400217    90.0  105.0   
1  [120.0, 130.0)   267.0  122.318352             3.002468   120.0  120.0   
2  [130.0, 139.0)   211.0  131.924171             2.698136   130.0  130.0   
3  [139.0, 198.0)   208.0  150.572115            11.507804   139.0  140.0   
4             nan     1.0  198.000000                  NaN   198.0  198.0   

   Mediana    75%  Máximo  
0    110.0  113.0   119.0  
1    120.0  125.0   129.0  
2    130.0  134.0   138.0  
3    147.5  159.0   190.0  
4    198.0  198.0   198.0  


C:\Users\loren\AppData\Local\Temp\ipykernel_16064\2395192822.py:110: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



#### Presion arterial diastolica

In [9]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

max_diastolica = max(datos.datos['presion_arterial_diastolica'])
intervalos = {'masculino': [0, 80, 90, 120, max_diastolica], 'femenino': [0, 80, 90, 120, max_diastolica]}

def plot_all():
    # Boxplot crudo
    fig = px.box(datos.datos, y='presion_arterial_diastolica', title='Boxplot crudo de presión arterial diastólica')
    fig.update_layout(yaxis_title='Presión Arterial Diastólica')
    fig.show()
    
    # Histograma crudo
    fig = px.histogram(datos.datos, x='presion_arterial_diastolica', nbins=10, 
                       title='Histograma crudo de presión arterial diastólica', marginal='rug')
    fig.update_layout(xaxis_title='Presión Arterial Diastólica')
    fig.show()
    
    # Tabla descriptiva general
    tabla = datos.datos['presion_arterial_diastolica'].describe().reset_index()
    tabla.columns = ['Estadística', 'Valor']
    print(tabla)
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla[col] for col in tabla.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas de presión arterial diastólica')
    fig.show()
    
    # Tablas descriptivas por agrupaciones independientes
    agrupaciones = ['sexo', 'tipo_de_afiliacion', 
                    'escolaridad', 'diabetes mellitus', 'hipercolesterolemia', 'dislipidemia',
                    'no_diabeticos_POC_hba1c(%)_controlado',
    'diabeticos_POC_hba1c(%)_controlado', 
    'bajo_peso',
    'normo_peso',
    'sobrepeso',
    'obesidad',
    'perimetro_abdominal_normal',
    'perimetro_abdominal_aumentado',
    'perimetro_abdominal_incremento_sustancial',
    'tension_normal',
    'tension_elevada',
    'tension_alta_nivel_1',
    'tension_alta_nivel_2']
    for agrupacion in agrupaciones:
        tabla_agrupacion = datos.datos.groupby(agrupacion)['presion_arterial_diastolica'].describe().reset_index()
        tabla_agrupacion = tabla_agrupacion.rename(columns={
            'mean': 'Media', 'std': 'Desviación estándar', '50%': 'Mediana', 
            'min': 'Mínimo', 'max': 'Máximo'
        })
        print(tabla_agrupacion)
        fig = go.Figure(data=[go.Table(
            header=dict(values=list(tabla_agrupacion.columns), fill_color='paleturquoise', align='left'),
            cells=dict(values=[tabla_agrupacion[col] for col in tabla_agrupacion.columns], fill_color='lavender', align='left')
        )])
        fig.update_layout(title_text=f'Estadísticas Descriptivas por {agrupacion}')
        fig.show()
    
    # Tabla descriptiva por intervalos
    datos.datos['presion_arterial_diastolica_intervalo'] = pd.cut(
        datos.datos['presion_arterial_diastolica'], intervalos['masculino'], right=False
    ).astype(str)
    tabla_intervalos = datos.datos.groupby('presion_arterial_diastolica_intervalo')['presion_arterial_diastolica'].describe().reset_index()
    tabla_intervalos.columns = ['Intervalo', 'Cuenta', 'Media', 'Desviación estándar', 'Mínimo', '25%', 'Mediana', '75%', 'Máximo']
    print(tabla_intervalos)
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla_intervalos.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla_intervalos[col] for col in tabla_intervalos.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas por Intervalos de presión arterial diastólica')
    fig.show()
    
    # Boxplot por intervalos categóricos
    intervalos_diastolica = [0, 80, 90, 100, float('inf')]
    labels_diastolica = ['[0, 80)', '[80, 90)', '[90, 100)', '[100, inf)']
    datos.datos['presion_arterial_diastolica_intervalo'] = pd.cut(
        datos.datos['presion_arterial_diastolica'], bins=intervalos_diastolica, 
        labels=labels_diastolica, right=False, include_lowest=True
    )
    datos_filtrados_diastolica = datos.datos.dropna(subset=['presion_arterial_diastolica_intervalo'])
    datos_filtrados_diastolica['presion_arterial_diastolica_intervalo'] = pd.Categorical(
        datos_filtrados_diastolica['presion_arterial_diastolica_intervalo'], 
        categories=labels_diastolica, ordered=True
    )
    fig = px.box(datos_filtrados_diastolica, x='presion_arterial_diastolica_intervalo', 
                 y='presion_arterial_diastolica', title='Boxplot de Presión Arterial Diastólica por Intervalos')
    fig.update_layout(
        xaxis_title='Intervalo', yaxis_title='Presión Arterial Diastólica', 
        xaxis={'categoryorder': 'array', 'categoryarray': labels_diastolica}
    )
    fig.show()

plot_all()



  Estadística       Valor
0       count  952.000000
1        mean   76.311975
2         std    9.591572
3         min   50.000000
4         25%   70.000000
5         50%   78.000000
6         75%   81.000000
7         max  129.000000


        sexo  count      Media  Desviación estándar  Mínimo   25%  Mediana  \
0   femenino  330.0  75.657576             9.115988    50.0  70.0     78.0   
1  masculino  622.0  76.659164             9.824035    51.0  70.0     78.0   

     75%  Máximo  
0  80.00   100.0  
1  81.75   129.0  


  tipo_de_afiliacion  count      Media  Desviación estándar  Mínimo   25%  \
0       contributivo  427.0  76.625293             9.238627    51.0  70.0   
1               otro    1.0  70.000000                  NaN    70.0  70.0   
2  polizas de seguro    1.0  68.000000                  NaN    68.0  68.0   
3          prepagada    1.0  70.000000                  NaN    70.0  70.0   
4   regimen especial   57.0  74.771930             9.437799    57.0  69.0   
5         subsidiado  465.0  76.258065             9.936765    50.0  70.0   

   Mediana   75%  Máximo  
0     77.0  81.0   105.0  
1     70.0  70.0    70.0  
2     68.0  68.0    68.0  
3     70.0  70.0    70.0  
4     74.0  81.0    93.0  
5     79.0  80.0   129.0  


               escolaridad  count      Media  Desviación estándar  Mínimo  \
0               analfabeta   37.0  76.297297            10.723455    60.0   
1             especialista   18.0  75.000000             7.955760    60.0   
2         estudios maximos    1.0  84.000000                  NaN    84.0   
3                  maestro    1.0  90.000000                  NaN    90.0   
4     pre-escolar completa    8.0  80.375000             4.103570    72.0   
5   pre-escolar incompleta   26.0  75.884615             9.799294    60.0   
6        primaria completa  196.0  76.551020             9.909977    52.0   
7      primaria incompleta  218.0  76.009174             9.393100    50.0   
8              profesional   84.0  74.071429             9.000382    55.0   
9      secundaria completa  156.0  78.070513             8.842064    60.0   
10   secundaria incompleta  142.0  75.521127            10.756307    59.0   
11               tecnologo   65.0  76.723077             8.709743    60.0   

  diabetes mellitus  count      Media  Desviación estándar  Mínimo   25%  \
0                no  646.0  76.767802             9.692574    50.0  70.0   
1                si  306.0  75.349673             9.317283    54.0  70.0   

   Mediana   75%  Máximo  
0     78.0  81.0   129.0  
1     75.0  80.0   100.0  


...

  tension_elevada  count      Media  Desviación estándar  Mínimo   25%  \
0              no  805.0  77.448447             9.735118    50.0  70.0   
1              si  147.0  70.088435             5.588546    60.0  70.0   

   Mediana   75%  Máximo  
0     80.0  82.0   129.0  
1     70.0  75.0    79.0  


  tension_alta_nivel_1  count      Media  Desviación estándar  Mínimo   25%  \
0                   no  511.0  73.512720            11.092805    50.0  68.0   
1                   si  441.0  79.555556             6.047330    60.0  80.0   

   Mediana   75%  Máximo  
0     70.0  77.0   129.0  
1     80.0  82.0   101.0  


  tension_alta_nivel_2  count      Media  Desviación estándar  Mínimo   25%  \
0                   no  819.0  74.626374             7.973801    50.0  70.0   
1                   si  133.0  86.691729            11.941892    51.0  77.0   

   Mediana   75%  Máximo  
0     76.0  80.0   101.0  
1     90.0  93.0   129.0  


       Intervalo  Cuenta       Media  Desviación estándar  Mínimo    25%  \
0    [0.0, 80.0)   521.0   69.527831             5.887380    50.0   66.0   
1   [80.0, 90.0)   333.0   81.735736             2.464657    80.0   80.0   
2  [90.0, 120.0)    97.0   93.587629             4.506367    90.0   90.0   
3            nan     1.0  129.000000                  NaN   129.0  129.0   

   Mediana    75%  Máximo  
0     70.0   74.0    79.0  
1     80.0   83.0    89.0  
2     91.0   98.0   109.0  
3    129.0  129.0   129.0  


C:\Users\loren\AppData\Local\Temp\ipykernel_16064\2594469400.py:83: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



## Conteos

In [12]:
for agrupacion in ['lugar_de_procedencia', 'lugar_de_nacimiento','tipo_de_afiliacion','escolaridad']:
            conteo_ciudades = datos.datos[agrupacion].value_counts().head(10)  
            df_ciudades = conteo_ciudades.reset_index()
            df_ciudades.columns = [agrupacion, 'Conteo']  
            fig = px.bar(df_ciudades, x=agrupacion, y='Conteo', title=f'10 Conteos con mas Más Apariciones por {agrupacion}', 
                        color='Conteo', text='Conteo', color_continuous_scale='blues')
            fig.update_layout(xaxis_title=agrupacion, yaxis_title='Número de Apariciones', xaxis_tickangle=-45)
            fig.show()